# 第10課 - 生產環境中的AI代理

在本課程中，您將學習使用 Microsoft Agent Framework（Python）的AI代理<strong>生產模式</strong>。我們涵蓋：

- <strong>可觀察性</strong> — 為代理互動添加時間和日誌記錄
- <strong>評估</strong> — 使用評估代理來評分回應質量
- <strong>成本管理</strong> — 代幣優化與模型選擇策略

情境是幫助使用者規劃行程的<strong>旅行代理</strong>，在其上層疊監控和評估。


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 生產環境考量

將 AI 代理從原型階段推向生產環境，需要特別注意三大支柱：

1. <strong>可觀察性</strong> — 你需要能夠掌握代理的行為、所花時間，以及調用的工具。沒有追蹤與日誌記錄，就幾乎無法進行生產問題的除錯。

2. <strong>評估</strong> — 自動化的品質檢查確保代理的回應隨時間保持準確、完整且有用。評估代理可以根據定義的標準對回應進行打分。

3. <strong>成本管理</strong> — 代幣使用量直接影響成本。像是提示優化、模型選擇和緩存等策略，有助於在不犧牲品質的前提下控制支出。


## 建立一個可觀察代理人

我們定義了 travel 工具，並以計時方式包裝代理人呼叫，以便我們可以監控延遲。在生產環境中，您會整合 OpenTelemetry 或類似的追蹤後端。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 評估模式

一個常見的生產模式是使用第二個代理作為<strong>評估者</strong>。評估者根據預定義的標準（如完整性、準確性和有用性）對主要代理的回應進行評分。

這可以實現：
- 在回應到達使用者之前自動進行品質門檻檢查
- 當提示或模型更改時進行迴歸檢測
- 持續監控代理的表現


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## 成本管理策略

控制成本對於生產中的 AI 代理至關重要。以下是關鍵策略：

| 策略 | 說明 |
|---|---|
| <strong>提示優化</strong> | 保持系統指令簡潔。移除冗餘上下文以減少輸入的 token 數量。 |
    "| <strong>模型選擇</strong> | 對於簡單任務如分類或抽取，使用較小且成本較低的模型（例如 GPT-4.1-mini），並保留較大型模型處理複雜的推理。 |\n",
| <strong>快取</strong> | 快取工具結果和常見查詢，以避免重複呼叫 API。 |
| **Token 預算** | 設定 `max_tokens` 限制以防止回應過長。 |
| <strong>批次處理</strong> | 盡可能將多個用戶查詢合併成一個 API 呼叫。 |

實務上，分層的方法運作良好：將簡單請求導向快速且便宜的模型，只有將複雜查詢升級至功能更強大的模型。


## 摘要

在本課程中，您學會了如何：

1. <strong>為代理互動增加可觀察性</strong>，透過時序和日誌記錄，為追蹤和監控奠定基礎。
2. 使用評估代理 <strong>自動評估代理回應</strong>，評分完整性、準確度和有用性。
3. 透過提示優化、模型選擇、快取和令牌預算來 <strong>管理成本</strong>。

這些生產模式有助於確保您的 AI 代理在大規模環境中具備可靠性、可度量性和成本效益。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
